# Problem 2 — Clean, Validate & Structure
**Datahut Data Science Internship Assignment — adidas India Men's Footwear**

Takes `products_raw.csv` (from Problem 1) and produces `products_clean.csv` — a clean, validated dataset ready for analysis.

**Steps:** Load → Deduplicate → Fix types → Derive discount columns → Validate → Save


## 1. Load

In [1]:
import pandas as pd

df = pd.read_csv("products_raw.csv")
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
df.head()


Rows: 1671, Columns: ['product_id', 'name', 'sub_brand', 'url', 'sale_price', 'mrp']


,product_id,name,sub_brand,url,sale_price,mrp
0,B75806,Samba OG Shoes,Originals,https://www.adidas.co.in/samba-og-shoes/B75806...,10999.0,10999
1,ID8757,Galaxy 7 Running Shoes,Performance,https://www.adidas.co.in/galaxy-7-running-shoe...,4409.5,6299
2,B75807,Samba OG Shoes,Originals,https://www.adidas.co.in/samba-og-shoes/B75807...,10999.0,10999
3,KI0066,PURECHILL SLIDES,Sportswear,https://www.adidas.co.in/purechill-slides/KI00...,6599.0,6599
4,JH6206,Adizero EVO SL Shoes,Performance,https://www.adidas.co.in/adizero-evo-sl-shoes/...,15999.0,15999


In [2]:
# Check data types before cleaning
df.dtypes


product_id        str
name              str
sub_brand         str
url               str
sale_price    float64
mrp             int64
dtype: object

## 2. Remove duplicates

In [3]:
before = len(df)

dupe_ids  = df[df.duplicated(subset="product_id", keep=False)]
dupe_urls = df[df.duplicated(subset="url", keep=False)]
print(f"Rows with duplicate product_id: {len(dupe_ids)}")
print(f"Rows with duplicate url:        {len(dupe_urls)}")

dupe_ids.sort_values("product_id")


Rows with duplicate product_id: 0
Rows with duplicate url:        0


,product_id,name,sub_brand,url,sale_price,mrp


In [4]:
df = df.drop_duplicates(subset="product_id", keep="first").reset_index(drop=True)
df = df.drop_duplicates(subset="url", keep="first").reset_index(drop=True)

print(f"Before: {before} rows")
print(f"After:  {len(df)} rows")
print(f"Removed: {before - len(df)} duplicate(s)")


Before: 1671 rows
After:  1671 rows
Removed: 0 duplicate(s)


## 3. Fix price column types

The assignment brief shows prices on the live site as text like `₹10 999.00`, which would need stripping of the currency symbol and spaces. Our scraper extracted prices from the structured JSON data embedded in the page HTML, so the values arrived as plain numbers — but they're still stored as strings in the CSV. We convert them to `float` explicitly and verify.


In [5]:
df["sale_price"] = pd.to_numeric(df["sale_price"], errors="coerce")
df["mrp"]        = pd.to_numeric(df["mrp"],        errors="coerce")

print(df[["sale_price", "mrp"]].dtypes)
df[["sale_price", "mrp"]].describe()


sale_price    float64
mrp             int64
dtype: object


,sale_price,mrp
count,1671.000000,1671.000000
mean,4426.170856,6839.813884
std,3263.760167,3629.737168
min,649.500000,999.000000
25%,2399.500000,4599.000000
50%,3299.000000,5599.000000
75%,5499.500000,8599.000000
max,34999.000000,34999.000000


## 4. Derive discount columns

- `discount_amount` = MRP − Sale Price
- `discount_pct` = discount as % of MRP, rounded to 1 decimal place

For full-price items, both come out to 0 correctly.


In [6]:
df["discount_amount"] = df["mrp"] - df["sale_price"]
df["discount_pct"]    = ((df["discount_amount"] / df["mrp"]) * 100).round(1)

df[["name", "sale_price", "mrp", "discount_amount", "discount_pct"]].sample(5, random_state=1)


,name,sale_price,mrp,discount_amount,discount_pct
1373,Crag Step Shoes,6599.0,6599,0.0,0.0
1520,HYPERBOOST EDGE Running Shoes,19999.0,19999,0.0,0.0
942,ADILETTE SLIDES,3919.5,5599,1679.5,30.0
1335,Oakz Top Sneaker,2299.5,4599,2299.5,50.0
1014,Tor-Ace Run Shoes,2999.5,5999,2999.5,50.0


## 5. Validate

Three checks for things that should never appear in a correct dataset.


In [7]:
# Check 1: sale_price > mrp — logically impossible
invalid_price = df[df["sale_price"] > df["mrp"]]
print(f"Rows where sale_price > mrp: {len(invalid_price)}")
invalid_price


Rows where sale_price > mrp: 0


,product_id,name,sub_brand,url,sale_price,mrp,discount_amount,discount_pct


In [8]:
# Check 2: missing critical fields
missing_name  = df[df["name"].isna()  | (df["name"] == "")]
missing_price = df[df["sale_price"].isna() | df["mrp"].isna()]
missing_url   = df[df["url"].isna()   | (df["url"] == "")]

print(f"Missing name:  {len(missing_name)}")
print(f"Missing price: {len(missing_price)}")
print(f"Missing url:   {len(missing_url)}")


Missing name:  0
Missing price: 0
Missing url:   0


In [9]:
# Check 3: on sale vs full price split
on_sale    = (df["discount_amount"] > 0).sum()
full_price = (df["discount_amount"] == 0).sum()

print(f"On sale:    {on_sale}  ({on_sale / len(df):.1%})")
print(f"Full price: {full_price}  ({full_price / len(df):.1%})")


On sale:    1400  (83.8%)
Full price: 271  (16.2%)


In [10]:
# Print and save a summary validation report
report_lines = [
    "VALIDATION REPORT — products_clean.csv",
    "=" * 40,
    f"Total rows after dedup:     {len(df)}",
    f"sale_price > mrp (invalid): {len(invalid_price)}",
    f"Missing name:               {len(missing_name)}",
    f"Missing price:              {len(missing_price)}",
    f"Missing url:                {len(missing_url)}",
    f"On sale:                    {on_sale} ({on_sale/len(df):.1%})",
    f"Full price:                 {full_price} ({full_price/len(df):.1%})",
]
report = "\n".join(report_lines)
print(report)

with open("validation_report.txt", "w", encoding="utf-8") as f:
    f.write(report)


VALIDATION REPORT — products_clean.csv
Total rows after dedup:     1671
sale_price > mrp (invalid): 0
Missing name:               0
Missing price:              0
Missing url:                0
On sale:                    1400 (83.8%)
Full price:                 271 (16.2%)


## 6. Save

In [11]:
df.to_csv("products_clean.csv", index=False)
print(f"Saved products_clean.csv — {len(df)} rows, {len(df.columns)} columns.")
df.head()


Saved products_clean.csv — 1671 rows, 8 columns.


,product_id,name,sub_brand,url,sale_price,mrp,discount_amount,discount_pct
0,B75806,Samba OG Shoes,Originals,https://www.adidas.co.in/samba-og-shoes/B75806...,10999.0,10999,0.0,0.0
1,ID8757,Galaxy 7 Running Shoes,Performance,https://www.adidas.co.in/galaxy-7-running-shoe...,4409.5,6299,1889.5,30.0
2,B75807,Samba OG Shoes,Originals,https://www.adidas.co.in/samba-og-shoes/B75807...,10999.0,10999,0.0,0.0
3,KI0066,PURECHILL SLIDES,Sportswear,https://www.adidas.co.in/purechill-slides/KI00...,6599.0,6599,0.0,0.0
4,JH6206,Adizero EVO SL Shoes,Performance,https://www.adidas.co.in/adizero-evo-sl-shoes/...,15999.0,15999,0.0,0.0
